[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/11_sliding_window.ipynb)

# 🔴 Hard: Sliding Window Attention

Реализуйте **Sliding Window Attention** — локальный вариант attention для длинных последовательностей. Вместо связи каждой позиции со всеми остальными позиция читает только соседей в радиусе `window_size`.

Each position $i$ can only attend to positions $j$ where $|i - j| \le w$ (the window size). В этом упражнении окно **симметричное и не causal**: позиция видит до `w` токенов слева, себя и до `w` токенов справа. На границах последовательности окно естественно обрезается.

## От полного attention к локальному

Сначала scores вычисляются как обычно:

$$S = \frac{QK^T}{\sqrt{d_k}}, \qquad S \in \mathbb{R}^{B\times L\times L}.$$

Затем пары с $|i-j|>w$ получают $-\infty$ **до** softmax. Поэтому каждая строка нормализуется только по разрешённому локальному окну:

$$A_{ij}=\operatorname{softmax}_j(S_{ij}+M_{ij}),\qquad M_{ij}=\begin{cases}0,&|i-j|\le w,\\-\infty,&|i-j|>w.\end{cases}$$

Для `L=6`, `w=1` позиция 0 видит `[0,1]`, позиция 3 — `[2,3,4]`, позиция 5 — `[4,5]`. Число разрешённых ключей у центральной позиции равно `2w+1`, но у краёв меньше. При `w=0` каждая строка содержит единственный допустимый ключ — саму себя, поэтому output точно равен `V`.

> Маска в учебной реализации остаётся плотной `(L,L)`, поэтому сама функция всё ещё требует $O(L^2)$ памяти и вычислений. Реальное ускорение Sliding Window Attention появляется со sparse/block-local kernels, которые не материализуют запрещённые пары. Здесь проверяется корректная семантика локального окна.

### Signature
```python
def sliding_window_attention(Q, K, V, window_size):
    # Q, K, V: (batch, seq, d) → output: (batch, seq, d_v)
    # window_size: int — position i attends to [i-w, i+w]
```

### Rules
- Do **NOT** use sparse attention libraries
- Mask positions outside the window with `-inf`
- `window_size=0`: only self — output should equal V
- `window_size >= seq_len`: equivalent to full attention

## План реализации

1. Вычислите scaled dot-product scores формы `(B,L,L)`.
2. Создайте индексы строк и столбцов на устройстве входов; сравнение их расстояния даст broadcast-маску `(L,L)`.
3. Заполните scores вне окна отрицательной бесконечностью.
4. Примените softmax по оси ключей и умножьте веса на `V`.

## Быстрые самопроверки

- `window_size=0` возвращает `V` при любых Q и K.
- `window_size>=L` совпадает с полным scaled dot-product attention.
- Изменение далёких K/V не влияет на output позиции вне их окна.
- Output сохраняет `(B,L,D_v)`, а backward достигает Q, K и V.

## Частые ошибки

- Маскировать допустимое окно вместо внешней области.
- Использовать строгое `< w`, теряя позиции ровно на расстоянии `w`.
- Подставлять 0 вместо `-inf`: запрещённый ключ тогда всё равно получает вероятность.
- Считать окно causal и закрывать правых соседей, хотя контракт задания симметричный.

## Материалы для глубокого изучения

- [Longformer: The Long-Document Transformer](https://arxiv.org/abs/2004.05150) — сочетание локального скользящего окна и глобального attention.
- [Mistral 7B](https://arxiv.org/abs/2310.06825) — Sliding Window Attention в decoder-only LLM.
- [PyTorch FlexAttention](https://pytorch.org/blog/flexattention/) — построение эффективных вариантов attention через программируемую mask logic.

## Вопросы для самопроверки

1. Сколько ключей видит центральная позиция при радиусе `w`?
2. Почему плотная маска реализует правильную семантику, но не даёт ожидаемую асимптотику?
3. Чем симметричное окно этого задания отличается от causal sliding window в decoder-модели?

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import math

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def sliding_window_attention(Q, K, V, window_size):
    pass  # Replace this

In [ ]:
# 🧪 Debug
Q = torch.randn(1, 6, 8)
K = torch.randn(1, 6, 8)
V = torch.randn(1, 6, 8)

out = sliding_window_attention(Q, K, V, window_size=1)
if out is None:
    print("Implement sliding_window_attention, then rerun this cell.")
else:
    print("Output shape:", out.shape)  # (1, 6, 8)

    # window=0 should return V
    out0 = sliding_window_attention(Q, K, V, window_size=0)
    print("window=0 == V?", torch.allclose(out0, V, atol=1e-5))

In [ ]:
from torch_judge import check
check('sliding_window')